
# Data preparation before training

This notebook loads the survey extract and prepares model-ready features for a quick depression/social isolation risk screen.

- Input file: `Dataset/example.csv` (loaded from project root)
- Target column: `eurod`
- Feature shortlist: `age`, `hhsize`, `partnerinhh`, `chronic_mod`, `mobilityind`, `casp`, `bmi`, `adla`, `grossmotor`, `sphus`, `recall_1`, `smoking`, `eduyears_mod`
- Row budget: keep at most 100k sampled rows for faster iteration (tweak `MAX_ROWS` if needed)

Feel free to extend the feature list; all steps below will reuse whatever is defined in `FEATURE_COLUMNS`.


In [91]:

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", None)


In [92]:

# Paths and columns
PROJECT_ROOT = Path.cwd().resolve().parent  # assumes the notebook lives in the `notebook/` folder
DATA_PATH = PROJECT_ROOT / 'Dataset' / 'example.csv'
MAX_ROWS = 100_000  # limit rows for smoother experimentation

FEATURE_COLUMNS = [
    'age',
    'hhsize',
    'partnerinhh',
    'chronic_mod',
    'mobilityind',
    'casp',
    'bmi',
    'adla',
    'grossmotor',
    'sphus',
    'recall_1',
    'smoking',
    'eduyears_mod',
]
TARGET_COLUMN = 'eurod'

# Binary label config (set USE_BINARY_LABEL=False to stay with regression target)
USE_BINARY_LABEL = True
BINARY_THRESHOLD = 4

# Negative codes in the extract denote different types of missing values; convert them to NaN.
MISSING_SENTINELS = [-16, -15, -13, -12, -10, -9, -8, -7, -5, -4, -3, -2, -1]



## Inspect coded missing values


In [93]:

# Quick look at how many coded-missing values we have before cleaning
negative_codes = {
    col: sorted(raw_df.loc[raw_df[col] < 0, col].unique())
    for col in usecols
}
missing_overview = pd.DataFrame({
    'coded_negatives': {col: int((raw_df[col] < 0).sum()) for col in usecols},
    'native_nan': raw_df[usecols].isna().sum(),
    'total_rows': len(raw_df),
})

# Store list-valued codes safely in a single column
negative_codes_table = pd.Series(negative_codes, name='negative_codes').to_frame()
display(negative_codes_table)
display(missing_overview)


,negative_codes
age,[-15.0]
hhsize,[]
partnerinhh,[]
chronic_mod,"[-15, -13, -12]"
mobilityind,"[-15, -13, -12]"
casp,"[-16, -15, -13]"
bmi,"[-15.0, -13.0, -12.0, -3.0]"
adla,"[-15, -13, -12]"
grossmotor,"[-15, -13, -12]"
sphus,"[-15, -12]"


,coded_negatives,native_nan,total_rows
age,3,0,100000
hhsize,0,0,100000
partnerinhh,0,0,100000
chronic_mod,6251,0,100000
mobilityind,6413,0,100000
casp,14399,0,100000
bmi,9116,0,100000
adla,6338,0,100000
grossmotor,6413,0,100000
sphus,371,0,100000



## Clean up missing values


In [94]:

# Replace coded-missing values with NaN
clean_df = raw_df.replace(MISSING_SENTINELS, np.nan)

missing_after = pd.DataFrame({
    'missing_after_clean': clean_df[usecols].isna().sum(),
    'total_rows': len(clean_df),
})
missing_after


,missing_after_clean,total_rows
age,3,100000
hhsize,0,100000
partnerinhh,0,100000
chronic_mod,6251,100000
mobilityind,6413,100000
casp,14399,100000
bmi,9116,100000
adla,6338,100000
grossmotor,6413,100000
sphus,371,100000


## Допълнително почистване и ограничаване на екстремни стойности
Премахваме нелогични/извънредни стойности чрез диапазони по колони и клъцване на опашките по перцентили преди сплит и скалиране.


In [95]:
# Филтрирай/нулирай нелогични стойности и намали екстремите
# Границите са съобразени с емпиричните стойности в екстракта
domain_bounds = {
    'age': (40, 110),
    'hhsize': (1, 15),
    'partnerinhh': (1, 3),
    'chronic_mod': (0, 10),
    'mobilityind': (0, 4),
    'casp': (0, 48),
    'bmi': (10, 60),
    'adla': (0, 5),
    'grossmotor': (0, 4),
    'sphus': (1, 5),
    'recall_1': (0, 10),
    'smoking': (1, 5),
    'eduyears_mod': (0, 35),
    TARGET_COLUMN: (0, 12),  # Euro-D e 0-12 по скала
}

# Нелогичните стойности ги маркираме като липсващи (ще се импутират)
for col, (low, high) in domain_bounds.items():
    out_of_range = ~clean_df[col].between(low, high)
    if out_of_range.any():
        clean_df.loc[out_of_range, col] = np.nan

# Winsorize (клипване) по 1-ви и 99-ти перцентил в рамките на домейн границите
winsor_cols = FEATURE_COLUMNS + [TARGET_COLUMN]
quantiles = clean_df[winsor_cols].quantile([0.01, 0.99])

for col in winsor_cols:
    q_low, q_high = quantiles.loc[0.01, col], quantiles.loc[0.99, col]
    bound_low, bound_high = domain_bounds[col]
    low = max(bound_low, q_low) if pd.notna(q_low) else bound_low
    high = min(bound_high, q_high) if pd.notna(q_high) else bound_high
    clean_df[col] = clean_df[col].clip(low, high)

# След допълнителното чистене може да има нови липси в целта
before = len(clean_df)
clean_df = clean_df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)
after = len(clean_df)
print(f'Dropped {before - after} rows after plausibility/outlier cleaning; remaining: {after}')


Dropped 22002 rows after plausibility/outlier cleaning; remaining: 77998


In [96]:

# Drop rows without the target; keep an index to trace back if needed
clean_df = clean_df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)
print(f"Rows after dropping missing {TARGET_COLUMN}: {clean_df.shape}")

clean_df.describe(include='all')


Rows after dropping missing eurod: (77998, 14)


,age,eduyears_mod,hhsize,partnerinhh,sphus,chronic_mod,casp,eurod,adla,mobilityind,grossmotor,recall_1,bmi,smoking
count,77910.000000,71259.000000,77998.000000,77998.000000,77968.000000,77949.000000,72996.000000,77998.000000,77979.000000,77968.000000,77968.000000,77211.000000,75833.000000,72216.000000
mean,67.369832,11.036395,2.119554,1.562194,3.147163,1.205031,37.421694,2.416870,0.193462,0.532796,0.302393,5.226911,26.887039,4.356652
std,10.028988,4.208168,0.964797,0.899075,1.058654,1.221300,6.218799,2.236382,0.684871,0.934104,0.761010,1.791114,4.444213,1.469532
min,48.500000,1.000000,1.000000,1.000000,1.000000,0.000000,21.000000,0.000000,0.000000,0.000000,0.000000,0.000000,18.256319,1.000000
25%,59.500000,8.000000,2.000000,1.000000,2.000000,0.000000,33.000000,1.000000,0.000000,0.000000,0.000000,4.000000,23.808798,5.000000
50%,66.800003,11.000000,2.000000,1.000000,3.000000,1.000000,38.000000,2.000000,0.000000,0.000000,0.000000,5.000000,26.297577,5.000000
75%,74.599998,14.000000,2.000000,3.000000,4.000000,2.000000,42.000000,4.000000,0.000000,1.000000,0.000000,6.000000,29.387754,5.000000
max,91.300003,21.000000,6.000000,3.000000,5.000000,5.000000,48.000000,9.000000,5.000000,4.000000,4.000000,9.000000,40.626240,5.000000



## Split and preprocess


In [97]:

X = clean_df[FEATURE_COLUMNS].copy()
y = clean_df[TARGET_COLUMN].copy()

if USE_BINARY_LABEL == False:
    y = (y >= BINARY_THRESHOLD).astype(int)
    y.name = f"{TARGET_COLUMN}_binary"
else:
    y.name = TARGET_COLUMN

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if USE_BINARY_LABEL else None,
)

print(f"Training rows: {len(X_train)}; Test rows: {len(X_test)}")
if USE_BINARY_LABEL:
    print(f"Positive rate - train: {y_train.mean():.3f}; test: {y_test.mean():.3f}")


Training rows: 62398; Test rows: 15600
Positive rate - train: 2.417; test: 2.417


In [98]:

# Impute missing values and scale numeric features to [0, 1]
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[('num', numeric_transformer, FEATURE_COLUMNS)],
    remainder='drop',
)

X_train_array = preprocessor.fit_transform(X_train)
X_test_array = preprocessor.transform(X_test)

# Keep original indices so we can map back to raw rows later
X_train_prepared = pd.DataFrame(X_train_array, columns=FEATURE_COLUMNS, index=X_train.index)
X_test_prepared = pd.DataFrame(X_test_array, columns=FEATURE_COLUMNS, index=X_test.index)

# Attach the target column to get ready-to-train tables (aligned by original index)
train_ready = pd.concat([X_train_prepared, y_train], axis=1)
test_ready = pd.concat([X_test_prepared, y_test], axis=1)


In [99]:

# Quick preview of prepared train/test tables
print(f"Train ready shape: {train_ready.shape}")
print(f"Test ready shape: {test_ready.shape}")
display(train_ready.head(1000))
display(test_ready.head(1000))


Train ready shape: (62398, 14)
Test ready shape: (15600, 14)


,age,hhsize,partnerinhh,chronic_mod,mobilityind,casp,bmi,adla,grossmotor,sphus,recall_1,smoking,eduyears_mod,eurod
27952,0.301402,0.2,0.0,0.6,0.00,0.518519,0.390242,0.0,0.0,0.75,0.444444,1.0,0.55,1.0
71701,0.116822,0.0,1.0,0.0,0.00,0.629630,0.333275,0.0,0.0,0.75,0.666667,1.0,0.40,4.0
8760,0.598131,0.2,0.0,0.0,0.00,0.518519,0.070852,0.0,0.0,0.50,0.888889,1.0,0.15,4.0
59952,0.257009,0.0,1.0,0.0,0.25,0.370370,0.605000,0.2,0.0,0.75,0.777778,0.0,0.50,5.0
41402,0.306075,0.2,0.0,0.2,0.00,0.185185,0.232555,0.0,0.0,0.75,0.555556,0.0,0.10,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25276,0.871495,0.0,1.0,0.0,0.75,0.555556,0.450978,0.0,0.5,0.50,0.666667,1.0,0.45,0.0
64172,0.754673,0.2,0.0,0.0,0.00,0.851852,0.244368,0.0,0.0,0.50,0.222222,1.0,0.35,3.0
39965,0.689252,0.0,1.0,0.0,0.00,0.111111,0.574646,0.0,0.0,0.75,0.555556,1.0,0.50,7.0
72356,0.700935,0.2,0.0,0.0,0.00,0.777778,0.348922,0.0,0.0,0.75,0.222222,1.0,0.35,2.0


,age,hhsize,partnerinhh,chronic_mod,mobilityind,casp,bmi,adla,grossmotor,sphus,recall_1,smoking,eduyears_mod,eurod
28203,0.161215,0.4,1.0,0.2,0.00,0.666667,0.258306,0.0,0.00,0.50,0.444444,0.0,0.55,4.0
22310,0.707944,0.0,1.0,0.2,0.00,0.629630,0.165884,0.0,0.00,0.00,0.444444,1.0,0.60,1.0
36115,0.252336,0.2,1.0,0.0,0.00,0.740741,0.563393,0.0,0.00,0.00,0.888889,1.0,0.75,2.0
59503,0.511682,0.2,0.0,0.4,0.00,0.518519,0.762851,0.0,0.00,0.75,0.666667,1.0,0.85,1.0
43401,0.549065,0.2,0.0,0.8,0.25,0.962963,0.496256,0.4,0.25,0.50,0.555556,1.0,0.50,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13981,0.271028,0.2,0.0,0.4,0.00,0.666667,0.101154,0.0,0.00,0.75,0.777778,1.0,0.65,5.0
45669,0.939252,0.0,1.0,0.4,0.00,0.740741,0.514285,0.0,0.00,1.00,0.444444,1.0,0.35,4.0
43654,0.462617,0.2,0.0,0.2,0.00,0.555556,0.150798,0.0,0.00,0.50,0.555556,0.0,0.80,2.0
23614,0.105140,0.6,0.0,0.2,0.00,0.703704,0.468282,0.0,0.00,0.50,0.444444,1.0,0.80,1.0


In [103]:
orig = clean_df[FEATURE_COLUMNS + [TARGET_COLUMN]].reset_index(drop=True)

row0 = train_ready.iloc[0]
if row0.name >= len(orig):
    raise IndexError(f'Row index {row0.name} not in cleaned data (len={len(orig)}).')

# Обратно мащабиране (по MinMax от preprocessor)
orig_like = pd.DataFrame(preprocessor.named_transformers_['num']['scaler'].inverse_transform(
    [row0[FEATURE_COLUMNS]]
), columns=FEATURE_COLUMNS)
print(orig_like.T)
print('Raw eurod (cleaned source):', orig.iloc[row0.name][TARGET_COLUMN])


                      0
age           61.400002
hhsize         2.000000
partnerinhh    1.000000
chronic_mod    3.000000
mobilityind    0.000000
casp          35.000000
bmi           26.986000
adla           0.000000
grossmotor     0.000000
sphus          4.000000
recall_1       4.000000
smoking        5.000000
eduyears_mod  12.000000
Raw eurod: -10.0


In [101]:

# Optional: persist preprocessed data for downstream model training
SAVE_PROCESSED = True
output_dir = PROJECT_ROOT / 'Dataset' / 'processed'
output_dir.mkdir(exist_ok=True)

if SAVE_PROCESSED:
    train_out = output_dir / 'train_prepared.parquet'
    test_out = output_dir / 'test_prepared.parquet'

    train_ready.to_parquet(train_out, index=False)
    test_ready.to_parquet(test_out, index=False)
    print(f"Saved {train_out} and {test_out} (target column: {train_ready.columns[-1]})")
else:
    print("Skipping save. Set SAVE_PROCESSED=True to write parquet files.")


Saved /Users/apostolov31/Desktop/smart-choices-better-lives-humansignal/Dataset/processed/train_prepared.parquet and /Users/apostolov31/Desktop/smart-choices-better-lives-humansignal/Dataset/processed/test_prepared.parquet (target column: eurod)



## Notes
- All preprocessing lives in the `preprocessor` object; you can reuse it in a training pipeline (e.g., `Pipeline([('prep', preprocessor), ('model', estimator)])`).
- Update `FEATURE_COLUMNS` to try different subsets without changing the rest of the notebook.
- Data are scaled with `MinMaxScaler` to keep values in [0, 1]; adjust to another scaler if you switch models.
- Binary label is enabled by default (`USE_BINARY_LABEL=True`, `BINARY_THRESHOLD=4`); set it to False to keep the raw `eurod` target.
